# Day 43 · 全链路压测与成本核算

**配套讲义**: [`days/day-43.md`](../days/day-43.md) ｜ **需要 GPU（云机器）**

压出真实的吞吐与 P95 延迟；算出单位经济：每千次会话的 GPU + API + 基础设施成本，并回答那个终极问题 —— **这个 SaaS 打不打得平**。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w8.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 压测（在终端跑）

In [ ]:
print("""
    python scripts/loadtest.py --url http://localhost:8080/v1/chat \
        --n 200 --concurrency 10

建议做三组：c=1 / c=10 / c=20，找到拐点。
""")

## 2. 成本模型（今天的主产出）

In [ ]:
# 把压测数字填进来
QPS            = 5.2
PEAK_FACTOR    = 5          # 真实峰值是压测的几分之一？保守取 5
GPU_HOURLY     = 1.88       # ¥/h
SHOPIFY_CUT    = 0.15       # 平台抽成
SERVER_MONTHLY = 60         # 应用服务器 ¥/月

per_hour = QPS / PEAK_FACTOR * 3600
gpu_per_session = GPU_HOURLY / per_hour
print(f"每小时可服务会话 ≈ {per_hour:.0f}")
print(f"单会话 GPU 成本   ≈ ¥{gpu_per_session:.5f}")

for price, quota in ((29, 500), (99, 2000), (299, 8000)):
    gross = price * (1 - SHOPIFY_CUT)
    cost = gpu_per_session * quota
    margin = (gross - cost) / gross
    print(f"  套餐 ¥{price:>3}/月 配额 {quota:>4} 次 → 毛利 ¥{gross-cost:>7.2f} "
          f"毛利率 {margin:>6.1%}")

## 3. 敏感性分析：哪个变量最要命

把 GPU 单价、峰值系数、配额三个变量各动 ±30%，看毛利率怎么变。
**结论通常是：配额（用户用得越多你越亏）比 GPU 单价更敏感。**

In [ ]:
def margin(price, quota, gpu_hourly=1.88, peak=5, qps=5.2, cut=0.15):
    per_hour = qps / peak * 3600
    gpu = gpu_hourly / per_hour * quota
    gross = price * (1 - cut)
    return (gross - gpu) / gross

base = margin(99, 2000)
print(f"基线毛利率 {base:.1%}")
for label, kw in [("GPU 涨价 30%", {"gpu_hourly": 1.88 * 1.3}),
                  ("峰谷比更差(8)", {"peak": 8}),
                  ("用户用满配额×1.5", {"quota": 3000})]:
    print(f"  {label:18s} → {margin(99, 2000, **kw):.1%}")

## 验收清单

- [ ] `reports/cost_model.md` 已产出，含 GPU + API + 基础设施三项成本
- [ ] P95 延迟和 QPS 是**压出来的真实数字**，不是估的
- [ ] 给出了定价建议与毛利测算
- [ ] **能正面回答「这个 SaaS 打不打得平」，以及如果不平，调哪三个旋钮**

**卡住了？** 回看 [`days/day-43.md`](../days/day-43.md) 第五节「容易踩的坑」。

> **明天**：`days/day-44.md` —— 安全、合规与可靠性